# 肺炎 X 光分类：一键运行全部实验

这个 notebook 会依次完成：

1. **主实验**：5 个模型（CNN、ResNet50、EfficientNet-B0、DenseNet121、ViT-B/16）× 3 个随机种子。每个种子会重新按患者划分验证集，结果报告"均值 ± 标准差"。
2. **消融实验**：在从零训练的 CNN 上逐项去掉改进（数据增强、类别加权、BatchNorm、分辨率、训练轮数、打乱顺序），外加一个 "v1 原始配方" 和一个 ResNet50 线性探针。
3. **后处理（不需要训练）**：Grad-CAM、5 模型集成、概率校准（温度缩放 / Platt 缩放）、汇总表格和图。

**使用方法**

1. 菜单 **代码执行程序 → 更改运行时类型 → GPU**（T4 即可，L4 / A100 更快）。
2. 检查下面第一个代码单元里的配置，主要是数据位置 `DRIVE_DATA_DIR`。
3. 菜单 **代码执行程序 → 全部运行**。授权挂载 Google Drive 后就不用管了。

**预计用时**：A100 上实测训练共 36 分钟（另需几分钟准备数据）；T4 预计需要 3–4 小时。

**断线怎么办**：每个训练完成后都会立刻保存到 Drive。断线或 GPU 额度用完后，重新连接再 **全部运行** 一次，已完成的部分会自动跳过，从断点继续。

建议先把 `PLAN` 设成 `"smoke"` 跑一遍（约 10 分钟，每个训练只跑几步），确认数据路径和环境都没问题，再改回 `"full"`。

In [ ]:
# ===== 配置 =====
REPO = "https://github.com/weirdquantum/pneumonia-xray-classification.git"
BRANCH = "main"

DATA_SOURCE = "drive"                        # "drive"：数据在 Google Drive；"kaggle"：用 kagglehub 自动下载
DRIVE_DATA_DIR = "/content/drive/MyDrive"    # 包含 train/ 和 test/ 文件夹的 Drive 路径（也会自动查找下面一两层）
OUTPUT_DIR = "/content/drive/MyDrive/pneumonia_v3"  # 所有结果保存在这里

PLAN = "full"            # "smoke"：几分钟跑通全流程做检查；"full"：正式实验
SEEDS = [42, 43, 44]     # 主实验的随机种子；只想跑一个种子就改成 [42]

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path

def sh(cmd, cwd=None):
    """Run a shell command, stream its output, stop the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ 没有检测到 GPU：请在 代码执行程序 → 更改运行时类型 中选择 GPU，否则训练会非常慢。")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_DIR = Path(OUTPUT_DIR) / ("smoke" if PLAN == "smoke" else "")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("结果保存到:", RUN_DIR)

In [ ]:
REPO_DIR = Path("/content/repo")
if (REPO_DIR / ".git").exists():
    sh(f"git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}", cwd=REPO_DIR)
else:
    sh(f"git clone -q -b {BRANCH} {REPO} {REPO_DIR}")
sh("git log --oneline -1", cwd=REPO_DIR)
sh("pip install -q -e .", cwd=REPO_DIR)

In [ ]:
def find_raw_root(base):
    """Folder that contains train/{NORMAL,PNEUMONIA} and test/{NORMAL,PNEUMONIA}, searched up to two levels down."""
    base = Path(base)
    for pattern in ("", "*", "*/*"):
        for cand in sorted(base.glob(pattern)) if pattern else [base]:
            if "__MACOSX" in str(cand):
                continue
            if all((cand / s / c).is_dir() for s in ("train", "test") for c in ("NORMAL", "PNEUMONIA")):
                return cand
    raise FileNotFoundError(f"在 {base} 下找不到 train/ 和 test/ 文件夹，请检查 DRIVE_DATA_DIR")

DATA_DIR = Path("/content/data")
CACHE_DIR = Path(OUTPUT_DIR) / "data_cache"   # 缓存好的图像也存在 Drive，下次直接复用
if (CACHE_DIR / "images.npy").exists():
    print("复用 Drive 上的图像缓存")
    DATA_DIR.mkdir(exist_ok=True)
    for f in ("images.npy", "manifest.csv", "data_report.json"):
        shutil.copy(CACHE_DIR / f, DATA_DIR / f)
else:
    if DATA_SOURCE == "kaggle":
        sh("pip install -q kagglehub")
        import kagglehub
        base = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
    else:
        base = DRIVE_DATA_DIR
    raw = find_raw_root(base)
    print("原始数据:", raw)
    local_raw = Path("/content/raw")
    if not local_raw.exists():  # 先复制到本地磁盘，比直接读 Drive 快得多
        local_raw.mkdir()
        for s in ("train", "val", "test"):
            if (raw / s).is_dir():
                print("复制", s, "...", flush=True)
                shutil.copytree(raw / s, local_raw / s)
    sh(f"python scripts/prepare_data.py --raw {local_raw} --out {DATA_DIR} --workers {os.cpu_count()}", cwd=REPO_DIR)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    for f in ("images.npy", "manifest.csv", "data_report.json"):
        shutil.copy(DATA_DIR / f, CACHE_DIR / f)
print(open(DATA_DIR / "data_report.json").read())

## 运行全部实验

这一步耗时最长。每个训练完成后会立刻保存到 Drive；如果中途断线，重新 **全部运行** 即可续跑。

In [ ]:
t0 = time.time()
try:
    sh(f"python scripts/experiments.py --plan {PLAN} --runs-root {RUN_DIR / 'runs'} --results-dir {RUN_DIR} "
       f"--data-dir {DATA_DIR} --work-dir /content/work --seeds {' '.join(map(str, SEEDS))}", cwd=REPO_DIR)
    print(f"全部完成，用时 {(time.time() - t0) / 3600:.1f} 小时")
except RuntimeError:
    # Keep going so the cells below still show and package whatever finished.
    print("\n⚠️ 有步骤失败，失败项列在上面输出的最后几行。下面仍会展示和打包已完成的结果；重新全部运行会从断点续跑。")

## 结果

In [ ]:
from IPython.display import Image, Markdown, display

titles = {"main": "主实验（均值 ± 标准差）", "first_seed": f"种子 {SEEDS[0]}（含 95% 置信区间）",
          "calibration": "概率校准（测试集）", "ablation": "消融实验"}
for name, title in titles.items():
    f = RUN_DIR / "results" / f"{name}.md"
    if f.exists():
        display(Markdown(f"### {title}\n\n" + f.read_text()))
for fig in sorted((RUN_DIR / "figures").glob("*.png")):
    display(Markdown(f"**{fig.name}**"))
    display(Image(str(fig), width=900))

In [ ]:
# 打包结果（不含模型权重），保存到 Drive 并下载到本地
from google.colab import files

zip_path = Path(OUTPUT_DIR) / f"pneumonia_v3_{PLAN}_results.zip"
staging = Path("/content/results_staging")
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree(RUN_DIR, staging, ignore=shutil.ignore_patterns("*.pt", "data_cache", "smoke", "*.zip"))
shutil.make_archive(str(zip_path.with_suffix("")), "zip", staging)
print(f"已保存: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
files.download(str(zip_path))

## 下一步

把下载的 zip 解压到本地项目根目录，里面的 `runs/`、`results/`、`figures/` 会覆盖同名文件夹，然后提交即可。也可以在本地重新生成汇总表：

```bash
python scripts/report.py --runs-root runs --out .
```

5 个模型（第一个种子）的权重保存在 Drive 的 `pneumonia_v3/runs/main/*/seed42/best.pt`，需要的话可以单独下载。